# Batch clean-remount verification for all 15 P4b shards

Use a CPU session with Internet enabled. Attach **Version 1** of every private shard dataset `thestonedape/task-aware-eeg2text-p4b-f{fold}-s{seed}`, plus task-segmented protocol v1, task-segmented schedule v1, and the exact launch-authorization dataset. Enable `GITHUB_TOKEN` and `P4B_FULL_LAUNCH_SHA256`. This notebook verifies all 15 shards without deserializing checkpoints, freezes their deterministic registry, and keeps partial scientific inspection disabled.

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
WORKTREE = Path('/kaggle/working/SemKey')
ASKPASS = Path('/kaggle/working/git_askpass.py')
OUTPUT = Path('/kaggle/working/task-aware-eeg2text-p4b-full-shard-verifications')
EXPECTED_UNITS = [(fold, seed) for fold in range(5) for seed in (20260717, 20260718, 20260719)]
EXPECTED_SHARDS = [f'p4b-f{fold}-s{seed}' for fold, seed in EXPECTED_UNITS]
PIN_PATHS = {
    'runner_source_sha256': 'evaluation/run_task_segmented_full_shard.py',
    'adapter_source_sha256': 'project_adapters/task_segmented_objective.py',
    'task_treatment_pilots_source_sha256': 'project_adapters/task_treatment_pilots.py',
    'shard_verifier_source_sha256': 'evaluation/verify_task_segmented_full_shard_artifact.py',
    'aggregator_source_sha256': 'evaluation/aggregate_task_segmented_full_shards.py',
    'decision_engine_source_sha256': 'evaluation/decide_task_segmented_objective.py',
    'execution_notebook_sha256': 'kaggle/run_task_segmented_full_shard.ipynb',
    'shard_clean_remount_verification_notebook_sha256': 'kaggle/verify_task_segmented_full_shard_artifact.ipynb',
    'complete_matrix_aggregation_notebook_sha256': 'kaggle/aggregate_task_segmented_full_shards.ipynb',
}

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

def is_sha256(value):
    return isinstance(value, str) and re.fullmatch(r'[0-9a-f]{64}', value) is not None

secrets = UserSecretsClient()
def required_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    assert value and value.strip(), f'Enable the private Kaggle secret {name}'
    return value.strip()

LAUNCH_SHA256 = required_secret('P4B_FULL_LAUNCH_SHA256').lower()
assert is_sha256(LAUNCH_SHA256)
launch_candidates = sorted(set(glob.glob(
    '/kaggle/input/**/task_segmented_full_launch_authorization.json', recursive=True
)))
assert len(launch_candidates) == 1, ('Attach exactly one launch authorization dataset', launch_candidates)
LAUNCH_PATH = Path(launch_candidates[0])
assert LAUNCH_PATH.is_file() and not LAUNCH_PATH.is_symlink()
assert digest(LAUNCH_PATH) == LAUNCH_SHA256, 'launch authorization secret/file SHA mismatch'
launch = json.loads(LAUNCH_PATH.read_text(encoding='utf-8'))
base_fields = {
    'schema_version', 'status', 'full_execution_contract_sha256', 'project_commit',
    'runtime_environment', 'authorized_shard_ids', 'full_training_authorized',
    'checkpoint_evaluation_authorized', 'confirmation_evaluation_authorized',
    'scientific_decision_permitted_after_complete_matrix_only',
    'partial_result_scientific_inspection_permitted',
    'official_validation_rows_read', 'official_validation_used_for_confirmation',
    'held_out_test_rows_read', 'held_out_test_accessed',
}
assert set(launch) == base_fields | set(PIN_PATHS)
assert launch['schema_version'] == 1 and launch['status'] == 'authorized_for_full_p4b_launch'
assert launch['authorized_shard_ids'] == EXPECTED_SHARDS
assert launch['runtime_environment'] == {
    'python': '3.12.13', 'numpy': '2.0.2', 'torch': '2.10.0+cu128',
    'torch_cuda': '12.8', 'device': 'cuda:0', 'minimum_cuda_device_count': 1,
    'selected_cuda_device_index': 0, 'selected_cuda_device_name': 'Tesla T4',
    'selected_cuda_compute_capability': [7, 5], 'cublas_workspace_config': ':4096:8',
    'deterministic_algorithms_required': True,
    'full_scientific_cpu_execution_permitted': False,
    'runtime_fingerprint_bound_to_shard_and_resume': True,
}
assert launch['full_training_authorized'] is True
assert launch['checkpoint_evaluation_authorized'] is True and launch['confirmation_evaluation_authorized'] is True
assert launch['scientific_decision_permitted_after_complete_matrix_only'] is True
for field in ('partial_result_scientific_inspection_permitted', 'official_validation_rows_read', 'official_validation_used_for_confirmation', 'held_out_test_rows_read', 'held_out_test_accessed'):
    assert launch[field] is False, field
PROJECT_COMMIT = launch['project_commit']
assert isinstance(PROJECT_COMMIT, str) and re.fullmatch(r'[0-9a-f]{40}', PROJECT_COMMIT)
assert is_sha256(launch['full_execution_contract_sha256'])
assert all(is_sha256(launch[key]) for key in PIN_PATHS)


In [ ]:
github_token = required_secret('GITHUB_TOKEN')
if WORKTREE.exists():
    shutil.rmtree(WORKTREE)
ASKPASS.write_text(
    "#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n",
    encoding='utf-8', newline='\n',
)
os.chmod(ASKPASS, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': str(ASKPASS), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token, 'PYTHONDONTWRITEBYTECODE': '1'})
try:
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(WORKTREE)], check=True, env=clone_env)
finally:
    if ASKPASS.exists():
        ASKPASS.unlink()
    del github_token, clone_env
subprocess.run(['git', '-C', str(WORKTREE), 'checkout', '--detach', PROJECT_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(WORKTREE), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PROJECT_COMMIT
assert digest(WORKTREE / 'evaluation/task_segmented_full_execution_contract.json') == launch['full_execution_contract_sha256']
for key, relative in PIN_PATHS.items():
    path = WORKTREE / relative
    assert path.is_file() and not path.is_symlink() and digest(path) == launch[key], key

def assert_clean_git():
    status = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'status', '--porcelain=v1', '--untracked-files=all', '--ignored=matching'], text=True
    ).strip()
    assert status == '', ('Git worktree is not clean', status)
    submodules = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'submodule', 'status', '--recursive'], text=True
    ).splitlines()
    assert all(line.startswith(' ') for line in submodules), ('Git submodule drift', submodules)

assert_clean_git()
test_env = os.environ.copy()
test_env['PYTHONDONTWRITEBYTECODE'] = '1'
subprocess.run([
    sys.executable, '-B', '-m', 'unittest',
    'evaluation.test_task_segmented_full_execution_contract',
    'evaluation.test_verify_task_segmented_full_shard_artifact',
], check=True, cwd=WORKTREE, env=test_env)
assert_clean_git()
print({'project_commit': actual_commit, 'local_launch_pins': 'PASS', 'regressions': 'PASS', 'git_clean': True})


In [ ]:
PROTOCOL_REQUIRED = {
    'batch_grid_feasibility.csv', 'candidate_pools.csv', 'confirmation_donors.csv',
    'outer_split_assignments.csv', 'protocol_registry.json', 'pseudo_groups.csv',
    'text_group_folds.csv', 'task_segmented_protocol_report.json',
    'protocol_freeze_run_metadata.json', 'task_segmented_objective_contract.json',
}
SCHEDULE_REQUIRED = {
    'trial_catalog.csv', 'schedule_indices.u32le', 'schedule_units.csv',
    'schedule_audit.csv', 'task_segmented_training_schedule_manifest.json',
    'task_segmented_training_schedule_report.json',
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json', 'schedule_freeze_run_metadata.json',
}
def exact_roots(marker, required):
    roots = []
    for marker_path in glob.glob('/kaggle/input/**/' + marker, recursive=True):
        root = Path(marker_path).parent
        try:
            names = {path.name for path in root.iterdir()}
        except OSError:
            continue
        if names == required and not root.is_symlink() and all(not path.is_symlink() for path in root.iterdir()):
            roots.append(root)
    return sorted(set(roots))
protocol_roots = exact_roots('task_segmented_protocol_report.json', PROTOCOL_REQUIRED)
schedule_roots = exact_roots('schedule_freeze_run_metadata.json', SCHEDULE_REQUIRED)
assert len(protocol_roots) == len(schedule_roots) == 1, {'protocol_v1': protocol_roots, 'schedule_v1': schedule_roots}
PROTOCOL_ROOT, SCHEDULE_ROOT = protocol_roots[0], schedule_roots[0]
assert 'task-aware-eeg2text-task-segmented-protocol' in PROTOCOL_ROOT.parts
assert 'task-aware-eeg2text-task-segmented-schedule' in SCHEDULE_ROOT.parts

shard_roots = {}
for manifest_path in glob.glob('/kaggle/input/**/full_shard_manifest.json', recursive=True):
    path = Path(manifest_path)
    if path.is_symlink():
        continue
    try:
        manifest = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        continue
    shard_id = manifest.get('shard_id')
    if shard_id not in EXPECTED_SHARDS:
        continue
    root = path.parent
    expected_slug = f'task-aware-eeg2text-{shard_id}'
    assert expected_slug in root.parts, ('Shard mounted from unexpected dataset slug', shard_id, root)
    assert shard_id not in shard_roots, ('Duplicate shard dataset', shard_id)
    shard_roots[shard_id] = root
assert set(shard_roots) == set(EXPECTED_SHARDS), ('Attach all 15 exact Version-1 shard datasets', sorted(shard_roots))
print({'protocol_root_v1': str(PROTOCOL_ROOT), 'schedule_root_v1': str(SCHEDULE_ROOT), 'shard_count_v1': len(shard_roots)})


In [ ]:
def verify_all_and_freeze_registry():
    if OUTPUT.exists():
        shutil.rmtree(OUTPUT)
    report_root = OUTPUT / 'reports'
    report_root.mkdir(parents=True)
    entries = []
    report_hashes = {}
    runtime_hashes = set()
    for fold, seed in EXPECTED_UNITS:
        shard_id = f'p4b-f{fold}-s{seed}'
        root = shard_roots[shard_id]
        manifest_sha256 = digest(root / 'full_shard_manifest.json')
        report_path = report_root / f'{shard_id}_verification_report.json'
        source_id = f'kaggle-dataset-thestonedape-task-aware-eeg2text-{shard_id}-version-1'
        subprocess.run([
            sys.executable, '-B', str(WORKTREE / 'evaluation/verify_task_segmented_full_shard_artifact.py'),
            '--artifact-root', str(root), '--expected-manifest-sha256', manifest_sha256,
            '--expected-contract-sha256', launch['full_execution_contract_sha256'],
            '--expected-launch-authorization-sha256', LAUNCH_SHA256,
            '--protocol-root', str(PROTOCOL_ROOT), '--schedule-root', str(SCHEDULE_ROOT),
            '--preserved-source-id', source_id, '--output-report', str(report_path),
        ], check=True, cwd='/kaggle/working', env=test_env)
        report = json.loads(report_path.read_text(encoding='utf-8'))
        assert report['status'] == 'pass' and report['outer_fold'] == fold and report['training_seed'] == seed
        assert report['preserved_source_id'] == source_id
        assert report['full_shard_manifest_sha256'] == manifest_sha256
        assert report['runtime_fingerprint_verified'] is True
        assert report['partial_scientific_decision_permitted'] is False
        assert report['official_validation_used_for_confirmation'] is False
        assert report['held_out_test_accessed'] is False and report['checkpoint_deserialized'] is False
        runtime_hashes.add(report['runtime_fingerprint_sha256'])
        report_sha256 = digest(report_path)
        report_hashes[shard_id] = report_sha256
        entries.append({
            'shard_id': shard_id, 'outer_fold': fold, 'training_seed': seed,
            'dataset_slug': f'thestonedape/task-aware-eeg2text-{shard_id}',
            'dataset_version': 1, 'preserved_source_id': source_id,
            'full_shard_manifest_sha256': manifest_sha256,
            'verification_report_sha256': report_sha256,
        })
    assert len(entries) == 15 and all(entry['dataset_version'] == 1 for entry in entries)
    assert len(runtime_hashes) == 1, ('Runtime fingerprint differs across shards', runtime_hashes)
    runtime_fingerprint_sha256 = next(iter(runtime_hashes))
    registry = {
        'schema_version': 1, 'status': 'frozen_after_all_p4b_shards_preserved',
        'full_execution_contract_sha256': launch['full_execution_contract_sha256'],
        'launch_authorization_sha256': LAUNCH_SHA256,
        'partial_scientific_decision_permitted': False,
        'held_out_test_accessed': False, 'shards': entries,
    }
    registry_path = OUTPUT / 'frozen_shard_registry.json'
    registry_path.write_bytes((json.dumps(registry, indent=2, sort_keys=True) + '\n').encode('utf-8'))
    registry_sha256 = digest(registry_path)
    metadata = {
        'schema_version': 1, 'status': 'pass', 'project_commit': PROJECT_COMMIT,
        'launch_authorization_sha256': LAUNCH_SHA256,
        'full_execution_contract_sha256': launch['full_execution_contract_sha256'],
        'frozen_shard_registry_sha256': registry_sha256,
        'runtime_fingerprint_sha256': runtime_fingerprint_sha256,
        'verified_shard_count': 15, 'verification_report_sha256': report_hashes,
        'checkpoint_deserialized': False,
        'partial_scientific_decision_permitted': False,
        'official_validation_used_for_confirmation': False,
        'held_out_test_accessed': False,
    }
    metadata_path = OUTPUT / 'verification_run_metadata.json'
    metadata_path.write_bytes((json.dumps(metadata, indent=2, sort_keys=True) + '\n').encode('utf-8'))
    assert set(path.name for path in OUTPUT.iterdir()) == {'reports', 'frozen_shard_registry.json', 'verification_run_metadata.json'}
    assert len(list(report_root.glob('*_verification_report.json'))) == 15
    assert_clean_git()
    return registry_sha256, runtime_fingerprint_sha256

try:
    registry_sha256, runtime_fingerprint_sha256 = verify_all_and_freeze_registry()
finally:
    if WORKTREE.exists():
        shutil.rmtree(WORKTREE)
    if ASKPASS.exists():
        ASKPASS.unlink()
assert not WORKTREE.exists() and not ASKPASS.exists()
print({
    'status': 'pass', 'verified_shards': 15, 'project_commit': PROJECT_COMMIT,
    'launch_authorization_sha256': LAUNCH_SHA256,
    'frozen_shard_registry_sha256': registry_sha256,
    'runtime_fingerprint_sha256': runtime_fingerprint_sha256,
    'partial_scientific_decision_permitted': False,
    'held_out_test_accessed': False, 'output_root': str(OUTPUT),
    'expected_private_dataset_slug': 'thestonedape/task-aware-eeg2text-p4b-full-shard-verifications',
    'expected_dataset_version': 1,
})
print('P4B ALL-15-SHARD CLEAN-REMOUNT VERIFICATION AND REGISTRY FREEZE: PASS')


After PASS, save the printed output root as private dataset `thestonedape/task-aware-eeg2text-p4b-full-shard-verifications`, **Version 1**. Copy the printed registry SHA-256 into the private Kaggle secret `P4B_FROZEN_REGISTRY_SHA256`. No scientific conclusion is allowed from this batch-verification output alone.